<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Gravitational_waves_3D_Space.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spacetime Curvature and Gravitational Waves: Mathematical Visualization

## Project Overview
This notebook contains a complete Python-based rendering pipeline for generating a high-quality mathematical visualization of general relativity concepts. The simulation focuses on the geometric nature of gravity, illustrating how mass distorts the fabric of spacetime and how accelerating masses generate ripples known as gravitational waves.

### Key Features
*   **3D Lattice Simulation**: A genuine three-dimensional wireframe lattice that responds vectorially to gravitational potential.
*   **Dynamic Quadrupole Radiation**: Implementation of rotating wave patterns that qualitatively represent the plus-polarization of gravitational waves from binary systems.
*   **Cinematic Rendering**: A multi-scene compositor that handles smooth crossfades, camera drifts, and stylized 'antique ink' aesthetics for educational storytelling.
*   **Spacetime Diagrams**: Integrated 4D worldline visualizations showing the relationship between spatial coordinates and time evolution.

## Scientific Background

### General Relativity and Curvature
In 1915, Albert Einstein proposed that gravity is not a force in the Newtonian sense, but a consequence of the curvature of spacetime. Massive objects change the geometry of the universe, and objects (and light) follow the 'straightest' possible paths (geodesics) through this curved manifold. This is often summarized by John Wheeler's famous phrase: "Spacetime tells matter how to move; matter tells spacetime how to curve."

### Gravitational Waves
Gravitational waves are disturbances in the curvature of spacetime, generated by accelerated masses, that propagate as waves outward from their source at the speed of light. The strongest sources are often binary systems containing compact objects like black holes or neutron stars.

Mathematically, for a source at a distance $r$, the gravitational wave amplitude $h$ is related to the second time derivative of the quadrupole moment $Q$ of the mass distribution:

$$h \sim \frac{2G}{c^4 r} \ddot{Q}$$

Due to the extreme stiffness of spacetime (reflected in the smallness of $G/c^4$), only the most massive and violent cosmic events produce detectable waves.

## Implementation Details

1.  **Metric Perturbation Approximation**: The visualization uses a simplified pedagogical model where the lattice nodes are displaced according to a potential field that saturates near the mass to maintain visual clarity.
2.  **Parchment Generation**: An aged paper effect is procedurally generated using multiple octaves of Perlin-like noise and Gaussian filters.
3.  **Cross-Backend Integration**: The project combines `NumPy` and `SciPy` for physics calculations with `Matplotlib` for 3D projection and `Pillow` (PIL) for high-level frame compositing.

## Sources and Further Reading

*   **Einstein, A. (1916)**: "The Foundation of the General Theory of Relativity." Annalen der Physik.
*   **Misner, C. W., Thorne, K. S., & Wheeler, J. A. (1973)**: *Gravitation*. W. H. Freeman (The definitive textbook on the subject).
*   **LIGO Scientific Collaboration**: "What are Gravitational Waves?" [ligo.org](https://www.ligo.org/science/GW-GW2.php).
*   **Schutz, B. (2009)**: *A First Course in General Relativity*. Cambridge University Press.

---
**Author**: Mugambi Ndwiga  
**Social**: @craftsandengineering  
**Repository**: Mathematical-video-animations-and-visualization


# Version 1

In [ ]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Curvature of Spacetime and Gravitational Waves
Repository: github.com/zombimann/Mathematical-video-animations-and-visualization
"""

import os
import io
import math
import urllib.request

import numpy as np
from scipy.ndimage import gaussian_filter
from PIL import Image, ImageDraw, ImageFilter, ImageFont

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import imageio

# ----------------------------------------------------------------------------
# 1. PARAMETERS  -- change anything here to re-tune the film
# ----------------------------------------------------------------------------

FRAME_W, FRAME_H = 1080, 1920          # YouTube Shorts, 9:16
FPS = 20

DURATIONS = dict(                       # seconds per scene -- edit freely
    hook=3.0,
    flat_space=5.0,
    single_mass=7.0,
    moving_mass_waves=7.5,
    binary_system=8.0,
    closing=1.8,
)

GRID_HALF_EXTENT = 6.0                  # spatial extent of the coordinate grid (x, y)
LATTICE_HALF_Z = 3.4                    # spatial extent of the grid above/below the plane
LATTICE_NXY = 9                         # lattice points along x and along y
LATTICE_NZ = 5                          # lattice points along z
SOFTENING = 1.05                        # Plummer-softening so the pull never blows up

MASS_A = 7.0                            # "weight" of body A (arbitrary visual units)
MASS_B = 3.2                            # "weight" of body B
ORBIT_RADIUS_SINGLE = 1.1               # wobble radius while A is alone
ORBIT_PERIOD_SINGLE = 6.0               # seconds per revolution
BINARY_SEPARATION = 2.6                 # separation once B arrives
BINARY_PERIOD = 4.2                     # seconds per mutual orbit

WAVE_SPEED = 5.1                        # how fast ripples expand outward
WAVE_WAVELENGTH = 2.1
WAVE_DECAY_LENGTH = 5.5                 # amplitude e-fold distance
WAVE_RING_WIDTH = 1.3                   # thickness of the visible wavefront band

CAMERA_ELEV = 16.0
CAMERA_AZIM_START = -58.0
CAMERA_AZIM_DRIFT = 10.0                # total degrees the camera slowly drifts

OUTPUT_PATH = "/mnt/user-data/outputs/spacetime_curvature_and_gravitational_waves.mp4"
TARGET_MAX_MB = 10.0

WORK_DIR = "/home/claude/_gw_render_cache"
os.makedirs(WORK_DIR, exist_ok=True)

# ----------------------------------------------------------------------------
# 2. PALETTE -- antique-ink colour language
# ----------------------------------------------------------------------------

PAPER_BASE_RGB = (231, 214, 178)
PAPER_TINT_RGB = (120, 90, 55)

INK_DARK = "#2b1d0e"        # primary ink (headers, captions)
INK_MED = "#5b3a1e"         # secondary ink (sub-labels)
GRID_INK = "#6b4a28"        # the spatial grid itself
CARD_FILL = (236, 222, 190, 96)    # translucent parchment card -- subtle, secondary
CARD_EDGE = (66, 45, 22, 110)

BODY_A_COLOR = "#7a2418"    # deep iron-gall red-brown wax seal
BODY_B_COLOR = "#1d3c52"    # blue-black ink
WAVE_COLOR_A = "#9a3a22"
WAVE_COLOR_B = "#274d66"

WATERMARK_TEXT = "\u00a9 Mugambi Ndwiga / @craftsandengineering"

# ----------------------------------------------------------------------------
# 3. FONTS -- handwritten / calligraphic, fetched once from Google Fonts
# ----------------------------------------------------------------------------

FONT_DIR = "/home/claude/fonts"
os.makedirs(FONT_DIR, exist_ok=True)

FONT_SOURCES = {
    "Caveat-Regular.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/caveat/Caveat%5Bwght%5D.ttf",
    "Sacramento-Regular.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/sacramento/Sacramento-Regular.ttf",
}


def _ensure_fonts():
    """Download the calligraphic typefaces if they are not already cached,
    then register them with matplotlib. Falls back to matplotlib's built-in
    cursive style if the network is unavailable, so the film always renders."""
    for fname, url in FONT_SOURCES.items():
        dest = os.path.join(FONT_DIR, fname)
        if not os.path.exists(dest):
            try:
                urllib.request.urlretrieve(url, dest)
            except Exception:
                pass

    regular_path = os.path.join(FONT_DIR, "Caveat-Regular.ttf")
    sig_path = os.path.join(FONT_DIR, "Sacramento-Regular.ttf")

    hand_font, sig_font = "cursive", "cursive"
    if os.path.exists(regular_path):
        try:
            from fontTools.ttLib import TTFont
            from fontTools.varLib.instancer import instantiateVariableFont
            static_path = os.path.join(FONT_DIR, "Caveat-Static.ttf")
            if not os.path.exists(static_path):
                f = TTFont(regular_path)
                instantiateVariableFont(f, {"wght": 560}).save(static_path)
            fm.fontManager.addfont(static_path)
            hand_font = fm.FontProperties(fname=static_path).get_name()
        except Exception:
            fm.fontManager.addfont(regular_path)
            hand_font = fm.FontProperties(fname=regular_path).get_name()

    if os.path.exists(sig_path):
        fm.fontManager.addfont(sig_path)
        sig_font = fm.FontProperties(fname=sig_path).get_name()

    return hand_font, sig_font


HAND_FONT, SIGNATURE_FONT = _ensure_fonts()
matplotlib.rcParams["mathtext.fontset"] = "cm"   # crisp, classic math typesetting

print(f"Using handwriting font: {HAND_FONT} | signature font: {SIGNATURE_FONT}")


# ----------------------------------------------------------------------------
# 4. AGED PAPER BACKGROUND -- generated once, reused on every frame
# ----------------------------------------------------------------------------

def generate_parchment(w, h, seed=11):
    rng = np.random.default_rng(seed)

    def cloud(scale_div, octave_strength, blur):
        small = rng.random((max(2, h // scale_div), max(2, w // scale_div)))
        im = Image.fromarray((small * 255).astype(np.uint8)).resize((w, h), Image.BICUBIC)
        arr = np.asarray(im, dtype=np.float32) / 255.0
        if blur:
            arr = gaussian_filter(arr, sigma=blur)
        return (arr - arr.mean()) * octave_strength

    field = np.zeros((h, w), dtype=np.float32)
    field += cloud(9, 22.0, 3)
    field += cloud(40, 10.0, 1)
    field += cloud(140, 5.0, 0)
    field += (rng.random((h, w)).astype(np.float32) - 0.5) * 6.0

    base = np.array(PAPER_BASE_RGB, dtype=np.float32).reshape(1, 1, 3)
    tint = np.array(PAPER_TINT_RGB, dtype=np.float32).reshape(1, 1, 3)
    t = np.clip((field + 30) / 60.0, 0, 1)[..., None]
    rgb = base * (1 - 0.55 * t) + tint * (0.55 * t)

    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    cx, cy = w / 2, h / 2
    r = np.sqrt(((xx - cx) / (w * 0.62)) ** 2 + ((yy - cy) / (h * 0.62)) ** 2)
    vignette = np.clip(1.0 - 0.35 * np.clip(r - 0.55, 0, None) ** 1.6, 0.45, 1.0)
    rgb *= vignette[..., None]

    img = Image.fromarray(np.clip(rgb, 0, 255).astype(np.uint8))

    blot_layer = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(blot_layer)
    n_blots = rng.integers(4, 8)
    for _ in range(n_blots):
        bx, by = rng.uniform(0, w), rng.uniform(0, h)
        br = rng.uniform(min(w, h) * 0.05, min(w, h) * 0.16)
        draw.ellipse([bx - br, by - br, bx + br, by + br], fill=int(rng.uniform(25, 60)))
    blot_layer = blot_layer.filter(ImageFilter.GaussianBlur(radius=min(w, h) * 0.03))
    blot_rgb = Image.new("RGB", (w, h), PAPER_TINT_RGB)
    img = Image.composite(blot_rgb, img, blot_layer)

    return img.convert("RGBA")


PARCHMENT_BG = generate_parchment(FRAME_W, FRAME_H, seed=11)


# ----------------------------------------------------------------------------
# 5. TEXT & CARD RENDERING -- crisp, solid ink on translucent parchment cards
# ----------------------------------------------------------------------------

def _render_text_rgba(text, font_name, size_px, color, mathtext=False, dpi=200):
    """Render a single line/paragraph of text to a tightly-cropped RGBA image
    with a transparent background, using matplotlib's Agg backend for
    anti-aliased, crisp glyph rendering."""
    fig = plt.figure(figsize=(0.1, 0.1), dpi=dpi)
    fig.patch.set_alpha(0)
    fontprops = dict(fontsize=size_px * 72 / dpi, color=color)
    if not mathtext:
        fontprops["fontfamily"] = font_name
    txt = fig.text(0.5, 0.5, text, ha="center", va="center", **fontprops)
    fig.canvas.draw()
    bbox = txt.get_window_extent(renderer=fig.canvas.get_renderer())
    pad = 6
    w = int(bbox.width) + pad * 2
    h = int(bbox.height) + pad * 2
    plt.close(fig)

    fig = plt.figure(figsize=(w / dpi, h / dpi), dpi=dpi)
    fig.patch.set_alpha(0)
    fig.text(0.5, 0.5, text, ha="center", va="center", **fontprops)
    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    return Image.fromarray(buf)


_TEXT_CACHE = {}


def render_text_cached(text, font_name, size_px, color, mathtext=False):
    key = (text, font_name, size_px, color, mathtext)
    if key not in _TEXT_CACHE:
        _TEXT_CACHE[key] = _render_text_rgba(text, font_name, size_px, color, mathtext)
    return _TEXT_CACHE[key]


def rounded_card(w, h, radius=22, fill=CARD_FILL, edge=CARD_EDGE, edge_width=2):
    card = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(card)
    draw.rounded_rectangle([1, 1, w - 2, h - 2], radius=radius, fill=fill,
                            outline=edge, width=edge_width)
    return card


def paste_centered(base, overlay, cx, cy):
    base.alpha_composite(overlay, (int(cx - overlay.width / 2), int(cy - overlay.height / 2)))


def paste_xy(base, overlay, x, y):
    base.alpha_composite(overlay, (int(x), int(y)))


# ----------------------------------------------------------------------------
# 6. PHYSICS -- simplified, pedagogical curvature & gravitational-wave model
#    applied to a genuine 3-D lattice (a meshgrid in x, y AND z).
# ----------------------------------------------------------------------------

_lat_x = np.linspace(-GRID_HALF_EXTENT, GRID_HALF_EXTENT, LATTICE_NXY)
_lat_y = np.linspace(-GRID_HALF_EXTENT, GRID_HALF_EXTENT, LATTICE_NXY)
_lat_z = np.linspace(-LATTICE_HALF_Z, LATTICE_HALF_Z, LATTICE_NZ)
LAT_X0, LAT_Y0, LAT_Z0 = np.meshgrid(_lat_x, _lat_y, _lat_z, indexing="ij")

MAX_PULL = 1.65                         # inward displacement saturates here (visual units)


def gravitational_pull(X0, Y0, Z0, cx, cy, cz, mass, softening=SOFTENING):
    """How far each 3-D lattice node is pulled toward a mass at (cx,cy,cz)."""
    rx, ry, rz = X0 - cx, Y0 - cy, Z0 - cz
    r = np.sqrt(rx ** 2 + ry ** 2 + rz ** 2)
    r_soft = np.sqrt(rx ** 2 + ry ** 2 + rz ** 2 + softening ** 2)
    raw = mass / r_soft
    pull = MAX_PULL * raw / (MAX_PULL + raw)
    inv_r = 1.0 / np.maximum(r, 1e-6)
    return -rx * inv_r * pull, -ry * inv_r * pull, -rz * inv_r * pull


def wave_displacement(X0, Y0, Z0, cx, cy, t_emit, now, amplitude, quad_phi0=None):
    """A single travelling, decaying ripple emitted at t_emit from (cx, cy, 0)."""
    if now <= t_emit:
        zeros = np.zeros_like(X0)
        return zeros, zeros, zeros
    rx, ry, rz = X0 - cx, Y0 - cy, Z0
    r = np.sqrt(rx ** 2 + ry ** 2 + rz ** 2)
    front = WAVE_SPEED * (now - t_emit)
    band = np.exp(-((r - front) ** 2) / (2 * WAVE_RING_WIDTH ** 2))
    phase = 2 * np.pi * (r - front) / WAVE_WAVELENGTH
    decay = np.exp(-r / WAVE_DECAY_LENGTH)
    mag = amplitude * decay * band * np.sin(phase)
    if quad_phi0 is not None:
        phi = np.arctan2(ry, rx)
        mag = mag * np.cos(2.0 * (phi - quad_phi0))
    inv_r = 1.0 / np.maximum(r, 1e-6)
    return rx * inv_r * mag, ry * inv_r * mag, rz * inv_r * mag


def body_a_position(t):
    t_wobble = DURATIONS["hook"] + DURATIONS["flat_space"] + DURATIONS["single_mass"]
    t_binary = t_wobble + DURATIONS["moving_mass_waves"]
    if t < t_wobble:
        return (0.0, 0.0)
    if t < t_binary:
        phase = 2 * math.pi * (t - t_wobble) / ORBIT_PERIOD_SINGLE
        return (ORBIT_RADIUS_SINGLE * math.cos(phase), ORBIT_RADIUS_SINGLE * math.sin(phase) * 0.6)
    phase = 2 * math.pi * (t - t_binary) / BINARY_PERIOD
    r = BINARY_SEPARATION * MASS_B / (MASS_A + MASS_B)
    return (r * math.cos(phase), r * math.sin(phase))


def body_b_position(t):
    t_wobble = DURATIONS["hook"] + DURATIONS["flat_space"] + DURATIONS["single_mass"]
    t_binary = t_wobble + DURATIONS["moving_mass_waves"]
    if t < t_binary:
        return None
    phase = 2 * math.pi * (t - t_binary) / BINARY_PERIOD
    r = BINARY_SEPARATION * MASS_A / (MASS_A + MASS_B)
    return (-r * math.cos(phase), -r * math.sin(phase))


def binary_axis_angle(t_emit):
    """Orientation (radians) of the line joining the two bodies at t_emit --
    used to lock the rotating quadrupole wave pattern to the orbit phase."""
    ax_, ay_ = body_a_position(t_emit)

    # Capture the position value first to check for None safely before unpacking
    pos_b = body_b_position(t_emit)
    if pos_b is None:
        return 0.0

    bx_, by_ = pos_b
    return math.atan2(ay_ - by_, ax_ - bx_)


def scene_boundaries():
    keys = ["hook", "flat_space", "single_mass", "moving_mass_waves", "binary_system", "closing"]
    t = 0.0
    bounds = {}
    for k in keys:
        bounds[k] = (t, t + DURATIONS[k])
        t += DURATIONS[k]
    return bounds, t


SCENE_BOUNDS, TOTAL_DURATION = scene_boundaries()


def current_scene(t):
    for name, (t0, t1) in SCENE_BOUNDS.items():
        if t0 <= t < t1 or (name == "closing" and t >= t0):
            return name, t0, t1
    return "closing", SCENE_BOUNDS["closing"][0], SCENE_BOUNDS["closing"][1]


def smoothstep(x):
    x = min(max(x, 0.0), 1.0)
    return x * x * (3 - 2 * x)


BODY_RADIUS_A = 0.55
BODY_RADIUS_B = 0.40
WAVE_AMPLITUDE_SCALE = 0.55
EMIT_INTERVAL = WAVE_WAVELENGTH / WAVE_SPEED

_T_WOBBLE = DURATIONS["hook"] + DURATIONS["flat_space"] + DURATIONS["single_mass"]
_T_BINARY = _T_WOBBLE + DURATIONS["moving_mass_waves"]


def sum_wave_field(now, position_fn, t_start, mass, quadrupole=False):
    """Superpose ripples emitted at regular intervals from position_fn(t_emit),
    from t_start up to `now`, each travelling outward at WAVE_SPEED."""
    zeros = np.zeros_like(LAT_X0)
    if now <= t_start:
        return zeros, zeros, zeros
    max_age = (GRID_HALF_EXTENT * 1.6) / WAVE_SPEED + 4 * WAVE_RING_WIDTH / WAVE_SPEED
    dx = np.zeros_like(LAT_X0)
    dy = np.zeros_like(LAT_X0)
    dz = np.zeros_like(LAT_X0)
    t_emit = t_start
    amplitude = WAVE_AMPLITUDE_SCALE * (mass / MASS_A)
    while t_emit < now:
        if now - t_emit < max_age:
            # Defensive safety check: ensure the tracked target returns a valid position tuple
            pos = position_fn(t_emit)
            if pos is not None:
                ex, ey = pos
                phi0 = binary_axis_angle(t_emit) if quadrupole else None
                ddx, ddy, ddz = wave_displacement(LAT_X0, LAT_Y0, LAT_Z0, ex, ey,
                                                   t_emit, now, amplitude, quad_phi0=phi0)
                dx += ddx; dy += ddy; dz += ddz
        t_emit += EMIT_INTERVAL
    return dx, dy, dz


def field_and_bodies(t):
    scene, t0, t1 = current_scene(t)
    X, Y, Z = LAT_X0, LAT_Y0, LAT_Z0
    bodies = []

    if scene in ("hook", "flat_space", "closing"):
        return X, Y, Z, bodies, scene

    if scene == "single_mass":
        ramp = smoothstep((t - t0) / DURATIONS["single_mass"])
        ax_, ay_ = body_a_position(t)
        mass = MASS_A * ramp
        dx, dy, dz = gravitational_pull(LAT_X0, LAT_Y0, LAT_Z0, ax_, ay_, 0.0, mass)
        X, Y, Z = LAT_X0 + dx, LAT_Y0 + dy, LAT_Z0 + dz
        bodies = [(ax_, ay_, 0.0, BODY_RADIUS_A, BODY_A_COLOR)]
        return X, Y, Z, bodies, scene

    if scene == "moving_mass_waves":
        ax_, ay_ = body_a_position(t)
        dx, dy, dz = gravitational_pull(LAT_X0, LAT_Y0, LAT_Z0, ax_, ay_, 0.0, MASS_A)
        wx, wy, wz = sum_wave_field(t, body_a_position, _T_WOBBLE, MASS_A, quadrupole=False)
        X, Y, Z = LAT_X0 + dx + wx, LAT_Y0 + dy + wy, LAT_Z0 + dz + wz
        bodies = [(ax_, ay_, 0.0, BODY_RADIUS_A, BODY_A_COLOR)]
        return X, Y, Z, bodies, scene

    if scene == "binary_system":
        ax_, ay_ = body_a_position(t)
        pos_b = body_b_position(t)
        bx_, by_ = pos_b if pos_b is not None else (0.0, 0.0)

        dxa, dya, dza = gravitational_pull(LAT_X0, LAT_Y0, LAT_Z0, ax_, ay_, 0.0, MASS_A)
        dxb, dyb, dzb = gravitational_pull(LAT_X0, LAT_Y0, LAT_Z0, bx_, by_, 0.0, MASS_B)
        wxa, wya, wza = sum_wave_field(t, body_a_position, _T_WOBBLE, MASS_A, quadrupole=True)
        wxb, wyb, wzb = sum_wave_field(t, body_b_position, _T_BINARY, MASS_B, quadrupole=True)
        X = LAT_X0 + dxa + dxb + wxa + wxb
        Y = LAT_Y0 + dya + dyb + wya + wyb
        Z = LAT_Z0 + dza + dzb + wza + wzb

        bodies = [(ax_, ay_, 0.0, BODY_RADIUS_A, BODY_A_COLOR)]
        if pos_b is not None:
            bodies.append((bx_, by_, 0.0, BODY_RADIUS_B, BODY_B_COLOR))
        return X, Y, Z, bodies, scene

    return X, Y, Z, bodies, scene


# ----------------------------------------------------------------------------
# 7. THE 3D CURVATURE GRID
# ----------------------------------------------------------------------------

from matplotlib.colors import LinearSegmentedColormap, Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection

DISTORTION_CMAP = LinearSegmentedColormap.from_list(
    "aged_ink_distortion", ["#e7cf9c", "#cdab74", "#8a6238", "#3a2613", "#140d06"])
DISTORTION_NORM = Normalize(vmin=0.0, vmax=1.7)
AXIS_LIM = GRID_HALF_EXTENT + WAVE_AMPLITUDE_SCALE + 0.35
ZLIM = (-(GRID_HALF_EXTENT + WAVE_AMPLITUDE_SCALE + 0.35), GRID_HALF_EXTENT + WAVE_AMPLITUDE_SCALE + 0.35)

MAIN_REGION_W = FRAME_W
MAIN_REGION_H = 1300
MAIN_DPI = 130


def _new_3d_axes(elev, azim):
    fig = plt.figure(figsize=(MAIN_REGION_W / MAIN_DPI, MAIN_REGION_H / MAIN_DPI), dpi=MAIN_DPI)
    fig.patch.set_alpha(0)
    o = 0.16
    ax = fig.add_axes([-o, -o, 1 + 2 * o, 1 + 2 * o], projection="3d")
    ax.patch.set_alpha(0)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(-AXIS_LIM, AXIS_LIM)
    ax.set_ylim(-AXIS_LIM, AXIS_LIM)
    ax.set_zlim(*ZLIM)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    return fig, ax


def lattice_to_segments(X, Y, Z):
    P = np.stack([X, Y, Z], axis=-1)
    dist = np.sqrt((X - LAT_X0) ** 2 + (Y - LAT_Y0) ** 2 + (Z - LAT_Z0) ** 2)

    segs, vals = [], []
    for axis in (0, 1, 2):
        a = [slice(None)] * 3
        b = [slice(None)] * 3
        a[axis] = slice(None, -1)
        b[axis] = slice(1, None)
        seg = np.stack([P[tuple(a)], P[tuple(b)]], axis=-2).reshape(-1, 2, 3)
        val = (0.5 * (dist[tuple(a)] + dist[tuple(b)])).reshape(-1)
        segs.append(seg)
        vals.append(val)

    return np.concatenate(segs, axis=0), np.concatenate(vals, axis=0)


def render_grid_scene(X, Y, Z, bodies, azim, elev=CAMERA_ELEV, alpha=0.88):
    fig, ax = _new_3d_axes(elev, azim)
    segments, values = lattice_to_segments(X, Y, Z)
    colors = DISTORTION_CMAP(DISTORTION_NORM(values))
    colors[:, 3] = alpha
    order = np.argsort(values)
    lc = Line3DCollection(segments[order], colors=colors[order], linewidths=1.2)
    ax.add_collection3d(lc)
    fig.canvas.draw()
    lattice_buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    lattice_img = Image.fromarray(lattice_buf)

    if not bodies:
        return lattice_img

    fig, ax = _new_3d_axes(elev, azim)
    for (bx, by, bz, br, color) in bodies:
        u = np.linspace(0, 2 * np.pi, 24)
        v = np.linspace(0, np.pi, 16)
        sx = bx + br * np.outer(np.cos(u), np.sin(v))
        sy = by + br * np.outer(np.sin(u), np.sin(v))
        sz = bz + br * np.outer(np.ones_like(u), np.cos(v))
        ax.plot_surface(sx, sy, sz, color=color, alpha=0.97, linewidth=0,
                         antialiased=True, shade=True)
    fig.canvas.draw()
    bodies_buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    bodies_img = Image.fromarray(bodies_buf)

    lattice_img.alpha_composite(bodies_img)
    return lattice_img


# ----------------------------------------------------------------------------
# 8. THE FOURTH DIMENSION -- Spacetime diagram
# ----------------------------------------------------------------------------

ST_PANEL_DPI = 150


def render_spacetime_diagram(t_now, t_window_start, t_window_end, w_px, h_px):
    fig = plt.figure(figsize=(w_px / ST_PANEL_DPI, h_px / ST_PANEL_DPI), dpi=ST_PANEL_DPI)
    fig.patch.set_alpha(0)
    ax = fig.add_axes([0.20, 0.20, 0.76, 0.66])
    ax.set_facecolor((1, 1, 1, 0))

    taus = np.linspace(t_window_start, max(t_window_start + 0.001, t_now), 160)
    xa = [body_a_position(tt)[0] for tt in taus]
    xb_raw = [body_b_position(tt) for tt in taus]
    xb = [v[0] if v is not None else None for v in xb_raw]

    ax.plot(xa, taus, color=BODY_A_COLOR, linewidth=2.6, solid_capstyle="round")
    if any(v is not None for v in xb):
        xb_clean = [v for v in xb if v is not None]
        taus_b = [tt for tt, v in zip(taus, xb) if v is not None]
        ax.plot(xb_clean, taus_b, color=BODY_B_COLOR, linewidth=2.6, solid_capstyle="round")

    ax.axhline(t_now, color=INK_DARK, linewidth=1.1, linestyle=(0, (4, 3)), alpha=0.75)
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(t_window_start, t_window_end)

    for spine in ax.spines.values():
        spine.set_color(INK_DARK)
        spine.set_linewidth(1.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])

    fp = fm.FontProperties(family=HAND_FONT)
    ax.set_xlabel("space", fontproperties=fp, fontsize=14, color=INK_DARK, labelpad=2)
    ax.set_ylabel("time", fontproperties=fp, fontsize=14, color=INK_DARK, labelpad=2)
    fig.text(0.5, 0.92, "spacetime (4D)", fontproperties=fp, fontsize=13,
              color=INK_MED, ha="center", va="center")

    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    return Image.fromarray(buf)


# ----------------------------------------------------------------------------
# 9. LAYOUT
# ----------------------------------------------------------------------------

MARGIN = 40
HEADER_XY = (MARGIN, 26)
HEADER_WH = (FRAME_W - 2 * MARGIN, 110)

MAIN_TOP = HEADER_XY[1] + HEADER_WH[1] + 16
MAIN_XY = (0, MAIN_TOP)

INFO_TOP = MAIN_TOP + MAIN_REGION_H + 14
INFO_WH = ((FRAME_W - 3 * MARGIN) // 2, 190)
INFO_LEFT_XY = (MARGIN, INFO_TOP)
INFO_RIGHT_XY = (MARGIN * 2 + INFO_WH[0], INFO_TOP)

CAPTION_TOP = INFO_TOP + INFO_WH[1] + 14
CAPTION_WH = (FRAME_W - 2 * MARGIN, 110)

WATERMARK_HEIGHT = int(FRAME_H * 0.019)

TITLE_SIZE = 52
EQUATION_SIZE = 38
NOTE_SIZE = 24
CAPTION_SIZE = 42
HOOK_TITLE_SIZE = 80
HOOK_SUB_SIZE = 36
CREDIT_SIZE = 54
CREDIT_SUB_SIZE = 34

# ----------------------------------------------------------------------------
# 10. SCENE SCRIPT
# ----------------------------------------------------------------------------

SCENE_TITLE = {
    "flat_space": "Flat Spacetime",
    "single_mass": "Mass Curves Space",
    "moving_mass_waves": "Motion Makes Waves",
    "binary_system": "Two Bodies, One Dance",
}

SCENE_CAPTION = {
    "flat_space": "Empty space. No mass. No curvature.",
    "single_mass": "Drop in a mass, and the grid bends toward it.",
    "moving_mass_waves": "An accelerating mass makes spacetime itself ripple.",
    "binary_system": "Two masses, orbiting -- the ripples grow stronger and interfere.",
}

SCENE_EQUATION = {
    "flat_space": r"$\Phi = 0$",
    "single_mass": r"$\Phi(r) = -\dfrac{GM}{\sqrt{r^{2}+a^{2}}}$",
    "moving_mass_waves": r"$h(r,t)\,\sim\,\dfrac{2G}{c^{4}r}\,\ddot{Q}(t)$",
    "binary_system": r"$h_{+}\,\sim\,\cos(2\phi-2\omega t)$",
}

SCENE_EQUATION_NOTE = {
    "flat_space": "no mass, no curvature",
    "single_mass": "curvature grows with mass, fades with distance",
    "moving_mass_waves": "an accelerating mass radiates outgoing ripples",
    "binary_system": "an orbiting pair: a rotating quadrupole wave",
}

SCENE_ANALOGY = {
    "single_mass": "Picture a marble dimpling a stretched net.",
    "moving_mass_waves": "Like ripples spreading outward across a pond.",
}

HOOK_TITLE = "How Large Masses Affect Space"
HOOK_SUBTITLE = "a short story about gravity"

CREDIT_LINE_1 = "Made by Mugambi Ndwiga"
CREDIT_LINE_2 = "@craftsandengineering"
CREDIT_LINE_3 = "Link to code is in the Description"


# ----------------------------------------------------------------------------
# 11. CARD BUILDERS
# ----------------------------------------------------------------------------

def make_card_with_lines(w, h, lines, pad_top=18):
    card = rounded_card(w, h)
    y = pad_top
    for text, font_name, size, color, mathtext, gap_after in lines:
        txt_img = render_text_cached(text, font_name, size, color, mathtext=mathtext)
        if txt_img.width > w - 2 * MARGIN // 2:
            scale = (w - 40) / txt_img.width
            txt_img = txt_img.resize((int(txt_img.width * scale), int(txt_img.height * scale)))
        paste_centered(card, txt_img, w / 2, y + txt_img.height / 2)
        y += txt_img.height + gap_after
    return card


def build_header_card(scene):
    w, h = HEADER_WH
    card = rounded_card(w, h)
    txt_img = render_text_cached(SCENE_TITLE[scene], HAND_FONT, TITLE_SIZE, INK_DARK)
    if txt_img.width > w - 56:
        scale = (w - 56) / txt_img.width
        txt_img = txt_img.resize((int(txt_img.width * scale), int(txt_img.height * scale)))
    paste_centered(card, txt_img, w / 2, h / 2)
    return card


def build_left_info_card(scene):
    w, h = INFO_WH
    lines = [
        (SCENE_EQUATION[scene], HAND_FONT, EQUATION_SIZE, INK_DARK, True, 14),
        (SCENE_EQUATION_NOTE[scene], HAND_FONT, NOTE_SIZE, INK_MED, False, 0),
    ]
    return make_card_with_lines(w, h, lines, pad_top=16)


def _wrap_text(text, max_chars=22):
    words = text.split()
    lines, cur = [], ""
    for word in words:
        trial = (cur + " " + word).strip()
        if len(trial) > max_chars and cur:
            lines.append(cur)
            cur = word
        else:
            cur = trial
    if cur:
        lines.append(cur)
    return lines


def _fit_and_stack(card, imgs, w, h, gap, side_pad=36, top_pad=16):
    max_w = max(im.width for im in imgs)
    scale_w = min(1.0, (w - side_pad) / max_w)
    total_h = sum(im.height for im in imgs) + gap * (len(imgs) - 1)
    scale_h = min(1.0, (h - top_pad) / max(total_h, 1))
    scale = min(scale_w, scale_h)
    if scale < 0.999:
        imgs = [im.resize((max(1, int(im.width * scale)), max(1, int(im.height * scale))))
                for im in imgs]
    total_h = sum(im.height for im in imgs) + gap * (len(imgs) - 1)
    y = h / 2 - total_h / 2
    for im in imgs:
        paste_centered(card, im, w / 2, y + im.height / 2)
        y += im.height + gap
    return card


def build_right_info_card_text(scene):
    w, h = INFO_WH
    card = rounded_card(w, h)
    text = SCENE_ANALOGY.get(scene, "Spacetime is the stage on which gravity plays.")
    wrapped = _wrap_text(text, max_chars=20)
    imgs = [render_text_cached(line, HAND_FONT, 30, INK_MED) for line in wrapped]
    return _fit_and_stack(card, imgs, w, h, gap=8)


def build_right_info_card_spacetime(t, t0, t1):
    w, h = INFO_WH
    card = rounded_card(w, h)
    diagram = render_spacetime_diagram(t, t0, t1, w - 20, h - 16)
    paste_centered(card, diagram, w / 2, h / 2)
    return card


def build_caption_card(scene):
    w, h = CAPTION_WH
    card = rounded_card(w, h)
    wrapped = _wrap_text(SCENE_CAPTION[scene], max_chars=40)
    imgs = [render_text_cached(line, HAND_FONT, CAPTION_SIZE, INK_DARK) for line in wrapped]
    return _fit_and_stack(card, imgs, w, h, gap=6)


_HEADER_CARD_CACHE = {s: build_header_card(s) for s in SCENE_TITLE}
_LEFT_CARD_CACHE = {s: build_left_info_card(s) for s in SCENE_TITLE}
_RIGHT_TEXT_CARD_CACHE = {s: build_right_info_card_text(s) for s in SCENE_TITLE}
_CAPTION_CARD_CACHE = {s: build_caption_card(s) for s in SCENE_TITLE}


WATERMARK_FONT_SIZE = 24


def build_watermark():
    txt = render_text_cached(WATERMARK_TEXT, HAND_FONT, WATERMARK_FONT_SIZE, INK_DARK)
    if txt.height > WATERMARK_HEIGHT:
        scale = WATERMARK_HEIGHT / txt.height
        txt = txt.resize((max(1, int(txt.width * scale)), max(1, int(txt.height * scale))))
    out = txt.copy()
    r, g, b, a = out.split()
    a = a.point(lambda v: int(v * 0.6))
    out.putalpha(a)
    return out


WATERMARK_IMG = build_watermark()


def paste_watermark(base):
    x = FRAME_W - WATERMARK_IMG.width - MARGIN
    y = FRAME_H - WATERMARK_IMG.height - int(MARGIN * 0.6)
    paste_xy(base, WATERMARK_IMG, x, y)


# ----------------------------------------------------------------------------
# 12. HOOK & CLOSING
# ----------------------------------------------------------------------------

def render_hook_frame(t):
    bg = PARCHMENT_BG.copy()
    fade_in = smoothstep(t / 0.5)
    fade_out = smoothstep((DURATIONS["hook"] - t) / 0.5)
    alpha = min(fade_in, fade_out)

    lines = HOOK_TITLE.split("\n")
    imgs = [render_text_cached(line, HAND_FONT, HOOK_TITLE_SIZE, INK_DARK) for line in lines]
    total_h = sum(im.height for im in imgs) + 12 * (len(imgs) - 1)
    y = FRAME_H / 2 - total_h / 2 - 60
    for im in imgs:
        im2 = im.copy()
        r, g, b, a = im2.split()
        a = a.point(lambda v: int(v * alpha))
        im2.putalpha(a)
        paste_centered(bg, im2, FRAME_W / 2, y + im2.height / 2)
        y += im2.height + 12

    sub_img = render_text_cached(HOOK_SUBTITLE, SIGNATURE_FONT, HOOK_SUB_SIZE, INK_MED)
    sub2 = sub_img.copy()
    r, g, b, a = sub2.split()
    a = a.point(lambda v: int(v * alpha))
    sub2.putalpha(a)
    paste_centered(bg, sub2, FRAME_W / 2, y + 40)

    paste_watermark(bg)
    return bg


def render_closing_frame(t):
    bg = PARCHMENT_BG.copy()
    line1 = render_text_cached(CREDIT_LINE_1, HAND_FONT, CREDIT_SIZE, INK_DARK)
    line2 = render_text_cached(CREDIT_LINE_2, SIGNATURE_FONT, CREDIT_SIZE, INK_DARK)
    line3 = render_text_cached(CREDIT_LINE_3, HAND_FONT, CREDIT_SUB_SIZE, INK_MED)
    total_h = line1.height + line2.height + line3.height + 60
    y = FRAME_H / 2 - total_h / 2
    paste_centered(bg, line1, FRAME_W / 2, y + line1.height / 2); y += line1.height + 22
    paste_centered(bg, line2, FRAME_W / 2, y + line2.height / 2); y += line2.height + 38
    paste_centered(bg, line3, FRAME_W / 2, y + line3.height / 2)
    paste_watermark(bg)
    return bg


# ----------------------------------------------------------------------------
# 13. FULL FRAME COMPOSITOR
# ----------------------------------------------------------------------------

def camera_azimuth(t):
    return CAMERA_AZIM_START + CAMERA_AZIM_DRIFT * (t / TOTAL_DURATION)


def render_main_frame(t, scene, t0, t1):
    bg = PARCHMENT_BG.copy()

    X, Y, Zc, bodies, _ = field_and_bodies(t)
    graphic = render_grid_scene(X, Y, Zc, bodies, azim=camera_azimuth(t))
    if graphic.size != (MAIN_REGION_W, MAIN_REGION_H):
        graphic = graphic.resize((MAIN_REGION_W, MAIN_REGION_H))
    paste_xy(bg, graphic, MAIN_XY[0], MAIN_XY[1])

    paste_xy(bg, _HEADER_CARD_CACHE[scene], *HEADER_XY)
    paste_xy(bg, _LEFT_CARD_CACHE[scene], *INFO_LEFT_XY)

    if scene == "binary_system":
        right_card = build_right_info_card_spacetime(t, t0, t1)
    else:
        right_card = _RIGHT_TEXT_CARD_CACHE[scene]
    paste_xy(bg, right_card, *INFO_RIGHT_XY)

    paste_xy(bg, _CAPTION_CARD_CACHE[scene], MARGIN, CAPTION_TOP)
    paste_watermark(bg)
    return bg


def render_frame(t):
    scene, t0, t1 = current_scene(t)
    if scene == "hook":
        frame = render_hook_frame(t)
    elif scene == "closing":
        frame = render_closing_frame(t)
    else:
        frame = render_main_frame(t, scene, t0, t1)
    return np.asarray(frame.convert("RGB"), dtype=np.uint8)


# ----------------------------------------------------------------------------
# 14. RENDER THE FILM
# ----------------------------------------------------------------------------

def render_video(output_path=OUTPUT_PATH, fps=FPS, crf=27, preset="medium"):
    n_frames = int(round(TOTAL_DURATION * fps))
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    writer = imageio.get_writer(
        output_path, fps=fps, codec="libx264", macro_block_size=1,
        ffmpeg_params=["-crf", str(crf), "-preset", preset, "-pix_fmt", "yuv420p"],
    )
    t_render_start = __import__("time").time()
    for i in range(n_frames):
        t = i / fps
        writer.append_data(render_frame(t))
        if i % max(1, n_frames // 20) == 0:
            elapsed = __import__("time").time() - t_render_start
            print(f"  frame {i + 1:4d}/{n_frames}  (t={t:5.2f}s)  elapsed={elapsed:5.1f}s")
    writer.close()
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Saved {output_path}  ({size_mb:.2f} MB, {n_frames} frames, {TOTAL_DURATION:.1f}s @ {fps}fps)")
    return output_path, size_mb


if __name__ == "__main__":
    path, size_mb = render_video()
    if size_mb > TARGET_MAX_MB:
        print(f"File is {size_mb:.2f} MB (> {TARGET_MAX_MB} MB target) -- re-encoding with a higher CRF...")
        path, size_mb = render_video(crf=32)
    print(f"FINAL: {path}  ({size_mb:.2f} MB)")

    # ------------------------------------------------------------------------
    # 15. DISPLAY + DOWNLOAD
    # ------------------------------------------------------------------------
    from IPython.display import Video, FileLink, display

    display(Video(path, embed=True, html_attributes="controls loop", width=380))

    try:
        from google.colab import files as _colab_files
        print("Running in Colab -- starting download...")
        _colab_files.download(path)
    except ImportError:
        print("Download link:")
        display(FileLink(path))
        print(f"(file saved at: {path})")

# Version 2 : Refined

In [ ]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Curvature of Spacetime and Gravitational Waves
Repository: github.com/zombimann/Mathematical-video-animations-and-visualization
"""

import os
import io
import math
import urllib.request

import numpy as np
from scipy.ndimage import gaussian_filter
from PIL import Image, ImageDraw, ImageFilter, ImageFont

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import imageio

# ----------------------------------------------------------------------------
# 1. PARAMETERS  -- change anything here to re-tune the film
# ----------------------------------------------------------------------------

FRAME_W, FRAME_H = 1080, 1920          # YouTube Shorts, 9:16
FPS = 20

DURATIONS = dict(                       # seconds per scene -- edit freely
    flat_space=5.0,                     # includes the opening title overlay
    single_mass=8.0,
    moving_mass_waves=13.0,
    binary_system=16.0,
    closing=2.0,
)

# -- cinematic pacing for entrances and blends (all independent of scene
#    length, so the *response* to gravity always feels immediate even
#    though the overall film is slow and unhurried) --
ENTRY_DURATION_A = 2.4                  # body A's dramatic flight into frame
MASS_RAMP_A = 0.8                       # how fast curvature builds in as A arrives
WOBBLE_SPIRAL_DURATION = 1.2            # smooth spiral-out into the small orbit
ENTRY_DURATION_B = 2.8                  # body B's dramatic flight into frame
MASS_RAMP_B = 0.8
BINARY_BLEND_DURATION = 1.4             # blending A from solo wobble into the binary orbit
CARD_TRANSITION = 1.3                   # crossfade duration between scene cards

GRAPHIC_FADE_IN_DURATION = 1.8          # the grid itself breathes into view at t=0
HOOK_TEXT_FADE_IN_END = 1.1
HOOK_TEXT_HOLD_END = 3.0
HOOK_TEXT_FADE_OUT_END = 4.2
CARDS_FADE_START = 3.0
CARDS_FADE_DURATION = 1.4

GRID_HALF_EXTENT = 6.0                  # spatial extent of the coordinate grid (x, y)
LATTICE_HALF_Z = 3.4                    # spatial extent of the grid above/below the plane
LATTICE_NXY = 9                         # lattice points along x and along y
LATTICE_NZ = 5                          # lattice points along z
SOFTENING = 1.3                         # Plummer-softening -- gentle, not a sharp funnel

MASS_A = 7.0                            # "weight" of body A (arbitrary visual units)
MASS_B = 3.2                            # "weight" of body B
ORBIT_RADIUS_SINGLE = 1.1               # wobble radius while A is alone
ORBIT_PERIOD_SINGLE = 7.0               # seconds per revolution
BINARY_SEPARATION = 2.6                 # separation once B arrives
BINARY_PERIOD = 5.2                     # seconds per mutual orbit

WAVE_SPEED = 2.6                        # how fast ripples expand outward
WAVE_WAVELENGTH = 2.3
WAVE_DECAY_LENGTH = 5.2                 # amplitude e-fold distance
WAVE_RING_WIDTH = 1.6                   # thickness of the visible wavefront band -- soft

CAMERA_ELEV = 16.0
CAMERA_AZIM_START = -58.0
CAMERA_AZIM_DRIFT = 16.0                # total degrees the camera slowly, cinematically drifts

OUTPUT_PATH = "/mnt/user-data/outputs/spacetime_curvature_and_gravitational_waves.mp4"
TARGET_MAX_MB = 10.0

WORK_DIR = "/home/claude/_gw_render_cache"
os.makedirs(WORK_DIR, exist_ok=True)

# ----------------------------------------------------------------------------
# 2. PALETTE -- antique-ink colour language
# ----------------------------------------------------------------------------

PAPER_BASE_RGB = (231, 214, 178)
PAPER_TINT_RGB = (120, 90, 55)

INK_DARK = "#2b1d0e"        # primary ink (headers, captions)
INK_MED = "#5b3a1e"         # secondary ink (sub-labels)
GRID_INK = "#6b4a28"        # the spatial grid itself
CARD_FILL = (236, 222, 190, 96)    # translucent parchment card -- subtle, secondary
CARD_EDGE = (66, 45, 22, 110)

BODY_A_COLOR = "#7a2418"    # deep iron-gall red-brown wax seal
BODY_B_COLOR = "#1d3c52"    # blue-black ink
WAVE_COLOR_A = "#9a3a22"
WAVE_COLOR_B = "#274d66"

WATERMARK_TEXT = "\u00a9 Mugambi Ndwiga / @craftsandengineering"

# ----------------------------------------------------------------------------
# 3. FONTS -- handwritten / calligraphic, fetched once from Google Fonts
# ----------------------------------------------------------------------------

FONT_DIR = "/home/claude/fonts"
os.makedirs(FONT_DIR, exist_ok=True)

FONT_SOURCES = {
    "Caveat-Regular.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/caveat/Caveat%5Bwght%5D.ttf",
    "Sacramento-Regular.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/sacramento/Sacramento-Regular.ttf",
}


def _ensure_fonts():
    """Download the calligraphic typefaces if they are not already cached,
    then register them with matplotlib. Falls back to matplotlib's built-in
    cursive style if the network is unavailable, so the film always renders."""
    for fname, url in FONT_SOURCES.items():
        dest = os.path.join(FONT_DIR, fname)
        if not os.path.exists(dest):
            try:
                urllib.request.urlretrieve(url, dest)
            except Exception:
                pass

    regular_path = os.path.join(FONT_DIR, "Caveat-Regular.ttf")
    sig_path = os.path.join(FONT_DIR, "Sacramento-Regular.ttf")

    hand_font, sig_font = "cursive", "cursive"
    if os.path.exists(regular_path):
        try:
            from fontTools.ttLib import TTFont
            from fontTools.varLib.instancer import instantiateVariableFont
            static_path = os.path.join(FONT_DIR, "Caveat-Static.ttf")
            if not os.path.exists(static_path):
                f = TTFont(regular_path)
                instantiateVariableFont(f, {"wght": 560}).save(static_path)
            fm.fontManager.addfont(static_path)
            hand_font = fm.FontProperties(fname=static_path).get_name()
        except Exception:
            fm.fontManager.addfont(regular_path)
            hand_font = fm.FontProperties(fname=regular_path).get_name()

    if os.path.exists(sig_path):
        fm.fontManager.addfont(sig_path)
        sig_font = fm.FontProperties(fname=sig_path).get_name()

    return hand_font, sig_font


HAND_FONT, SIGNATURE_FONT = _ensure_fonts()
matplotlib.rcParams["mathtext.fontset"] = "cm"   # crisp, classic math typesetting

print(f"Using handwriting font: {HAND_FONT} | signature font: {SIGNATURE_FONT}")


# ----------------------------------------------------------------------------
# 4. AGED PAPER BACKGROUND -- generated once, reused on every frame
# ----------------------------------------------------------------------------

def generate_parchment(w, h, seed=11):
    rng = np.random.default_rng(seed)

    def cloud(scale_div, octave_strength, blur):
        small = rng.random((max(2, h // scale_div), max(2, w // scale_div)))
        im = Image.fromarray((small * 255).astype(np.uint8)).resize((w, h), Image.BICUBIC)
        arr = np.asarray(im, dtype=np.float32) / 255.0
        if blur:
            arr = gaussian_filter(arr, sigma=blur)
        return (arr - arr.mean()) * octave_strength

    field = np.zeros((h, w), dtype=np.float32)
    field += cloud(9, 22.0, 3)
    field += cloud(40, 10.0, 1)
    field += cloud(140, 5.0, 0)
    field += (rng.random((h, w)).astype(np.float32) - 0.5) * 6.0

    base = np.array(PAPER_BASE_RGB, dtype=np.float32).reshape(1, 1, 3)
    tint = np.array(PAPER_TINT_RGB, dtype=np.float32).reshape(1, 1, 3)
    t = np.clip((field + 30) / 60.0, 0, 1)[..., None]
    rgb = base * (1 - 0.55 * t) + tint * (0.55 * t)

    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    cx, cy = w / 2, h / 2
    r = np.sqrt(((xx - cx) / (w * 0.62)) ** 2 + ((yy - cy) / (h * 0.62)) ** 2)
    vignette = np.clip(1.0 - 0.35 * np.clip(r - 0.55, 0, None) ** 1.6, 0.45, 1.0)
    rgb *= vignette[..., None]

    img = Image.fromarray(np.clip(rgb, 0, 255).astype(np.uint8))

    blot_layer = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(blot_layer)
    n_blots = rng.integers(4, 8)
    for _ in range(n_blots):
        bx, by = rng.uniform(0, w), rng.uniform(0, h)
        br = rng.uniform(min(w, h) * 0.05, min(w, h) * 0.16)
        draw.ellipse([bx - br, by - br, bx + br, by + br], fill=int(rng.uniform(25, 60)))
    blot_layer = blot_layer.filter(ImageFilter.GaussianBlur(radius=min(w, h) * 0.03))
    blot_rgb = Image.new("RGB", (w, h), PAPER_TINT_RGB)
    img = Image.composite(blot_rgb, img, blot_layer)

    return img.convert("RGBA")


PARCHMENT_BG = generate_parchment(FRAME_W, FRAME_H, seed=11)


# ----------------------------------------------------------------------------
# 5. TEXT & CARD RENDERING -- crisp, solid ink on translucent parchment cards
# ----------------------------------------------------------------------------

def _render_text_rgba(text, font_name, size_px, color, mathtext=False, dpi=200):
    """Render a single line/paragraph of text to a tightly-cropped RGBA image
    with a transparent background, using matplotlib's Agg backend for
    anti-aliased, crisp glyph rendering."""
    fig = plt.figure(figsize=(0.1, 0.1), dpi=dpi)
    fig.patch.set_alpha(0)
    fontprops = dict(fontsize=size_px * 72 / dpi, color=color)
    if not mathtext:
        fontprops["fontfamily"] = font_name
    txt = fig.text(0.5, 0.5, text, ha="center", va="center", **fontprops)
    fig.canvas.draw()
    bbox = txt.get_window_extent(renderer=fig.canvas.get_renderer())
    pad = 6
    w = int(bbox.width) + pad * 2
    h = int(bbox.height) + pad * 2
    plt.close(fig)

    fig = plt.figure(figsize=(w / dpi, h / dpi), dpi=dpi)
    fig.patch.set_alpha(0)
    fig.text(0.5, 0.5, text, ha="center", va="center", **fontprops)
    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    return Image.fromarray(buf)


_TEXT_CACHE = {}


def render_text_cached(text, font_name, size_px, color, mathtext=False):
    key = (text, font_name, size_px, color, mathtext)
    if key not in _TEXT_CACHE:
        _TEXT_CACHE[key] = _render_text_rgba(text, font_name, size_px, color, mathtext)
    return _TEXT_CACHE[key]


def rounded_card(w, h, radius=22, fill=CARD_FILL, edge=CARD_EDGE, edge_width=2):
    card = Image.new("RGBA", (w, h), (0, 0, 0, 0))
    draw = ImageDraw.Draw(card)
    draw.rounded_rectangle([1, 1, w - 2, h - 2], radius=radius, fill=fill,
                            outline=edge, width=edge_width)
    return card


def paste_centered(base, overlay, cx, cy):
    base.alpha_composite(overlay, (int(cx - overlay.width / 2), int(cy - overlay.height / 2)))


def paste_xy(base, overlay, x, y):
    base.alpha_composite(overlay, (int(x), int(y)))


# ----------------------------------------------------------------------------
# 6. PHYSICS -- simplified, pedagogical curvature & gravitational-wave model
#    applied to a genuine 3-D lattice (a meshgrid in x, y AND z).
#
#    Curvature term  :  every lattice node is pulled toward each mass along
#                        the straight line joining them, by an amount that
#                        saturates with depth -- a 3-D, vector generalisation
#                        of the textbook "rubber sheet" embedding diagram.
#                            pull(r) = M / sqrt(r^2 + a^2),     r in 3-D
#    Wave term        :  each node additionally breathes in and out along
#                        that same radial line as a travelling, decaying
#                        ripple sourced at the body's retarded position --
#                        a qualitative stand-in for the quadrupole formula
#                        h ~ (2G / c^4 r) * d^2Q/dt^2. For the binary the
#                        ripple is further shaped by a rotating cos(2*phi)
#                        lobe pattern, echoing the true "+"-polarisation
#                        of quadrupole radiation.
# ----------------------------------------------------------------------------

_lat_x = np.linspace(-GRID_HALF_EXTENT, GRID_HALF_EXTENT, LATTICE_NXY)
_lat_y = np.linspace(-GRID_HALF_EXTENT, GRID_HALF_EXTENT, LATTICE_NXY)
_lat_z = np.linspace(-LATTICE_HALF_Z, LATTICE_HALF_Z, LATTICE_NZ)
LAT_X0, LAT_Y0, LAT_Z0 = np.meshgrid(_lat_x, _lat_y, _lat_z, indexing="ij")

MAX_PULL = 0.85                         # inward displacement saturates here -- subtle, elegant


def gravitational_pull(X0, Y0, Z0, cx, cy, cz, mass, softening=SOFTENING):
    """How far each 3-D lattice node is pulled toward a mass at (cx,cy,cz).
    Returns (dx, dy, dz) displacement components -- the grid bends in all
    three spatial directions, not just along a single height channel."""
    rx, ry, rz = X0 - cx, Y0 - cy, Z0 - cz
    r = np.sqrt(rx ** 2 + ry ** 2 + rz ** 2)
    r_soft = np.sqrt(rx ** 2 + ry ** 2 + rz ** 2 + softening ** 2)
    raw = mass / r_soft
    pull = MAX_PULL * raw / (MAX_PULL + raw)
    inv_r = 1.0 / np.maximum(r, 1e-6)
    return -rx * inv_r * pull, -ry * inv_r * pull, -rz * inv_r * pull


def wave_displacement(X0, Y0, Z0, cx, cy, t_emit, now, amplitude, quad_phi0=None):
    """A single travelling, decaying ripple emitted at t_emit from (cx, cy, 0),
    displacing nodes radially in 3-D. If quad_phi0 is given, the ripple is
    shaped into a rotating two-lobed (quadrupole) pattern instead of a
    plain expanding sphere -- the qualitatively correct shape for radiation
    from an orbiting pair."""
    if now <= t_emit:
        zeros = np.zeros_like(X0)
        return zeros, zeros, zeros
    rx, ry, rz = X0 - cx, Y0 - cy, Z0
    r = np.sqrt(rx ** 2 + ry ** 2 + rz ** 2)
    front = WAVE_SPEED * (now - t_emit)
    band = np.exp(-((r - front) ** 2) / (2 * WAVE_RING_WIDTH ** 2))
    phase = 2 * np.pi * (r - front) / WAVE_WAVELENGTH
    decay = np.exp(-r / WAVE_DECAY_LENGTH)
    mag = amplitude * decay * band * np.sin(phase)
    if quad_phi0 is not None:
        phi = np.arctan2(ry, rx)
        mag = mag * np.cos(2.0 * (phi - quad_phi0))
    inv_r = 1.0 / np.maximum(r, 1e-6)
    return rx * inv_r * mag, ry * inv_r * mag, rz * inv_r * mag


def smoothstep(x):
    x = min(max(x, 0.0), 1.0)
    return x * x * (3 - 2 * x)


def _lerp(p, q, frac):
    return (p[0] * (1 - frac) + q[0] * frac, p[1] * (1 - frac) + q[1] * frac)


# -- phase timeline (all derived from DURATIONS, nothing hand-tuned twice) --
T_FLAT_END = DURATIONS["flat_space"]
T_A_ARRIVE_END = T_FLAT_END + ENTRY_DURATION_A
T_SINGLE_END = T_FLAT_END + DURATIONS["single_mass"]
T_WOBBLE_START = T_SINGLE_END
T_WOBBLE_SPIRAL_END = T_WOBBLE_START + WOBBLE_SPIRAL_DURATION
T_WAVES_END = T_WOBBLE_START + DURATIONS["moving_mass_waves"]
T_BINARY_START = T_WAVES_END
T_B_ARRIVE_END = T_BINARY_START + ENTRY_DURATION_B
T_BINARY_BLEND_END = T_BINARY_START + BINARY_BLEND_DURATION
T_BINARY_END = T_BINARY_START + DURATIONS["binary_system"]

ENTRY_POINT_A = (-10.4, 7.2)             # body A flies in from far off-frame
ENTRY_POINT_B = (10.8, -7.6)             # body B flies in from the opposite side


def _wobble_pos(t):
    phase = 2 * math.pi * (t - T_WOBBLE_START) / ORBIT_PERIOD_SINGLE
    return (ORBIT_RADIUS_SINGLE * math.cos(phase), ORBIT_RADIUS_SINGLE * math.sin(phase) * 0.6)


def _binary_pos_a(t):
    phase = 2 * math.pi * (t - T_BINARY_START) / BINARY_PERIOD
    r = BINARY_SEPARATION * MASS_B / (MASS_A + MASS_B)
    return (r * math.cos(phase), r * math.sin(phase))


def _binary_pos_b(t):
    phase = 2 * math.pi * (t - T_BINARY_START) / BINARY_PERIOD
    r = BINARY_SEPARATION * MASS_A / (MASS_A + MASS_B)
    return (-r * math.cos(phase), -r * math.sin(phase))


def body_a_position(t):
    """Body A's position throughout the film -- a single continuous path:
    flies in dramatically from off-frame, settles, then smoothly spirals
    out into its small wobble orbit and later eases into the binary dance.
    No teleporting: every phase change blends continuously into the next."""
    if t < T_FLAT_END:
        return None
    if t < T_A_ARRIVE_END:
        frac = smoothstep((t - T_FLAT_END) / ENTRY_DURATION_A)
        return _lerp(ENTRY_POINT_A, (0.0, 0.0), frac)
    if t < T_WOBBLE_START:
        return (0.0, 0.0)
    if t < T_WOBBLE_SPIRAL_END:
        frac = smoothstep((t - T_WOBBLE_START) / WOBBLE_SPIRAL_DURATION)
        wx, wy = _wobble_pos(t)
        return (wx * frac, wy * frac)
    if t < T_BINARY_START:
        return _wobble_pos(t)
    if t < T_BINARY_BLEND_END:
        frac = smoothstep((t - T_BINARY_START) / BINARY_BLEND_DURATION)
        return _lerp(_wobble_pos(t), _binary_pos_a(t), frac)
    return _binary_pos_a(t)


def body_b_position(t):
    """Body B's position -- absent until the binary scene, then a dramatic
    entrance from off-frame, easing smoothly into its side of the orbit."""
    if t < T_BINARY_START:
        return None
    if t < T_B_ARRIVE_END:
        frac = smoothstep((t - T_BINARY_START) / ENTRY_DURATION_B)
        return _lerp(ENTRY_POINT_B, _binary_pos_b(t), frac)
    return _binary_pos_b(t)


def mass_a(t):
    """How much curvature Body A is currently exerting. Ramps in quickly as
    it arrives -- by the time it lands, space has (almost) finished bending,
    so the response reads as immediate rather than a slow creep."""
    if t < T_FLAT_END:
        return 0.0
    return MASS_A * smoothstep((t - T_FLAT_END) / MASS_RAMP_A)


def mass_b(t):
    if t < T_BINARY_START:
        return 0.0
    return MASS_B * smoothstep((t - T_BINARY_START) / MASS_RAMP_B)


def binary_axis_angle(t_emit):
    """Orientation (radians) of the line joining the two bodies at t_emit --
    used to lock the rotating quadrupole wave pattern to the orbit phase."""
    pos_a = body_a_position(t_emit)
    pos_b = body_b_position(t_emit)
    if pos_a is None or pos_b is None:
        return 0.0
    return math.atan2(pos_a[1] - pos_b[1], pos_a[0] - pos_b[0])


def scene_boundaries():
    keys = ["flat_space", "single_mass", "moving_mass_waves", "binary_system", "closing"]
    t = 0.0
    bounds = {}
    for k in keys:
        bounds[k] = (t, t + DURATIONS[k])
        t += DURATIONS[k]
    return bounds, t


SCENE_BOUNDS, TOTAL_DURATION = scene_boundaries()
SCENE_ORDER = ["flat_space", "single_mass", "moving_mass_waves", "binary_system", "closing"]


def current_scene(t):
    for name, (t0, t1) in SCENE_BOUNDS.items():
        if t0 <= t < t1 or (name == "closing" and t >= t0):
            return name, t0, t1
    return "closing", SCENE_BOUNDS["closing"][0], SCENE_BOUNDS["closing"][1]


def previous_scene(scene):
    idx = SCENE_ORDER.index(scene)
    return SCENE_ORDER[idx - 1] if idx > 0 else None


BODY_RADIUS_A = 0.55
BODY_RADIUS_B = 0.40
WAVE_AMPLITUDE_SCALE = 0.22              # subtle, elegant ripples -- not vigorous
EMIT_INTERVAL = WAVE_WAVELENGTH / WAVE_SPEED


def sum_wave_field(now, position_fn, t_start, mass, quadrupole=False):
    """Superpose ripples emitted at regular intervals from position_fn(t_emit),
    from t_start up to `now`, each travelling outward at WAVE_SPEED. Returns
    (dx, dy, dz) displacement arrays over the full 3-D lattice."""
    zeros = np.zeros_like(LAT_X0)
    if now <= t_start:
        return zeros, zeros, zeros
    max_age = (GRID_HALF_EXTENT * 1.6) / WAVE_SPEED + 4 * WAVE_RING_WIDTH / WAVE_SPEED
    dx = np.zeros_like(LAT_X0)
    dy = np.zeros_like(LAT_X0)
    dz = np.zeros_like(LAT_X0)
    t_emit = t_start
    amplitude = WAVE_AMPLITUDE_SCALE * (mass / MASS_A)
    while t_emit < now:
        if now - t_emit < max_age:
            pos = position_fn(t_emit)
            if pos is not None:
                phi0 = binary_axis_angle(t_emit) if quadrupole else None
                ddx, ddy, ddz = wave_displacement(LAT_X0, LAT_Y0, LAT_Z0, pos[0], pos[1],
                                                   t_emit, now, amplitude, quad_phi0=phi0)
                dx += ddx; dy += ddy; dz += ddz
        t_emit += EMIT_INTERVAL
    return dx, dy, dz


def field_and_bodies(t):
    """Returns (X, Y, Z, bodies, scene_name): the displaced 3-D lattice
    coordinates, ready to be drawn as a bent grid of lines, plus a list of
    (x, y, z, radius, color) tuples for the bodies sitting inside it. Driven
    entirely by continuous position/mass functions -- nothing branches on
    scene identity, so there is never a jump or teleport at a scene boundary."""
    scene, t0, t1 = current_scene(t)

    pos_a = body_a_position(t)
    pos_b = body_b_position(t)
    m_a = mass_a(t)
    m_b = mass_b(t)

    dx = np.zeros_like(LAT_X0)
    dy = np.zeros_like(LAT_X0)
    dz = np.zeros_like(LAT_X0)
    bodies = []

    if pos_a is not None:
        if m_a > 1e-3:
            ddx, ddy, ddz = gravitational_pull(LAT_X0, LAT_Y0, LAT_Z0, pos_a[0], pos_a[1], 0.0, m_a)
            dx += ddx; dy += ddy; dz += ddz
        bodies.append((pos_a[0], pos_a[1], 0.0, BODY_RADIUS_A, BODY_A_COLOR))

    if pos_b is not None:
        if m_b > 1e-3:
            ddx, ddy, ddz = gravitational_pull(LAT_X0, LAT_Y0, LAT_Z0, pos_b[0], pos_b[1], 0.0, m_b)
            dx += ddx; dy += ddy; dz += ddz
        bodies.append((pos_b[0], pos_b[1], 0.0, BODY_RADIUS_B, BODY_B_COLOR))

    if t > T_WOBBLE_START:
        quad = t > T_BINARY_START
        wxa, wya, wza = sum_wave_field(t, body_a_position, T_WOBBLE_START, MASS_A, quadrupole=quad)
        dx += wxa; dy += wya; dz += wza

    if t > T_BINARY_START:
        wxb, wyb, wzb = sum_wave_field(t, body_b_position, T_BINARY_START, MASS_B, quadrupole=True)
        dx += wxb; dy += wyb; dz += wzb

    X, Y, Z = LAT_X0 + dx, LAT_Y0 + dy, LAT_Z0 + dz
    return X, Y, Z, bodies, scene


# ----------------------------------------------------------------------------
# 7. THE 3D CURVATURE GRID -- the film's central visual
#    A genuine 3-D wireframe lattice (lines running along x, y AND z) whose
#    nodes are displaced by gravity_pull / wave_displacement above. Colour
#    encodes how far each strand of the lattice has been bent from its rest
#    position -- dark ink where space is most distorted, pale parchment
#    where it is essentially flat.
# ----------------------------------------------------------------------------

from matplotlib.colors import LinearSegmentedColormap, Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection

DISTORTION_CMAP = LinearSegmentedColormap.from_list(
    "aged_ink_distortion", ["#b8854a", "#96693a", "#6b4726", "#3a2613", "#140d06"])
DISTORTION_NORM = Normalize(vmin=0.0, vmax=0.95)
AXIS_LIM = GRID_HALF_EXTENT + WAVE_AMPLITUDE_SCALE + 0.25
ZLIM = (-(GRID_HALF_EXTENT + WAVE_AMPLITUDE_SCALE + 0.25), GRID_HALF_EXTENT + WAVE_AMPLITUDE_SCALE + 0.25)

MAIN_REGION_W = FRAME_W
MAIN_REGION_H = 1240
MAIN_DPI = 130


def _new_3d_axes(elev, azim):
    fig = plt.figure(figsize=(MAIN_REGION_W / MAIN_DPI, MAIN_REGION_H / MAIN_DPI), dpi=MAIN_DPI)
    fig.patch.set_alpha(0)
    o = 0.24   # overscan: crops matplotlib's default 3-D margin so the lattice
               # spans the full width of the frame and dominates the composition
    ax = fig.add_axes([-o, -o, 1 + 2 * o, 1 + 2 * o], projection="3d")
    ax.patch.set_alpha(0)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(-AXIS_LIM, AXIS_LIM)
    ax.set_ylim(-AXIS_LIM, AXIS_LIM)
    ax.set_zlim(*ZLIM)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    return fig, ax


def lattice_to_segments(X, Y, Z):
    """Vectorised conversion of a displaced (nx, ny, nz) lattice into the
    full set of line segments running along its three index directions,
    each tagged with a colour driven by local displacement magnitude."""
    P = np.stack([X, Y, Z], axis=-1)
    dist = np.sqrt((X - LAT_X0) ** 2 + (Y - LAT_Y0) ** 2 + (Z - LAT_Z0) ** 2)

    segs, vals = [], []
    for axis in (0, 1, 2):
        a = [slice(None)] * 3
        b = [slice(None)] * 3
        a[axis] = slice(None, -1)
        b[axis] = slice(1, None)
        seg = np.stack([P[tuple(a)], P[tuple(b)]], axis=-2).reshape(-1, 2, 3)
        val = (0.5 * (dist[tuple(a)] + dist[tuple(b)])).reshape(-1)
        segs.append(seg)
        vals.append(val)

    return np.concatenate(segs, axis=0), np.concatenate(vals, axis=0)


def render_grid_scene(X, Y, Z, bodies, azim, elev=CAMERA_ELEV, alpha=0.88):
    # -- pass 1: the bent 3-D lattice alone --
    fig, ax = _new_3d_axes(elev, azim)
    segments, values = lattice_to_segments(X, Y, Z)
    colors = DISTORTION_CMAP(DISTORTION_NORM(values))
    colors[:, 3] = alpha
    order = np.argsort(values)               # draw calmest strands first
    lc = Line3DCollection(segments[order], colors=colors[order], linewidths=1.2)
    ax.add_collection3d(lc)
    fig.canvas.draw()
    lattice_buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    lattice_img = Image.fromarray(lattice_buf)

    if not bodies:
        return lattice_img

    # -- pass 2: the bodies alone, same exact camera & axes limits --
    fig, ax = _new_3d_axes(elev, azim)
    for (bx, by, bz, br, color) in bodies:
        u = np.linspace(0, 2 * np.pi, 24)
        v = np.linspace(0, np.pi, 16)
        sx = bx + br * np.outer(np.cos(u), np.sin(v))
        sy = by + br * np.outer(np.sin(u), np.sin(v))
        sz = bz + br * np.outer(np.ones_like(u), np.cos(v))
        ax.plot_surface(sx, sy, sz, color=color, alpha=0.97, linewidth=0,
                         antialiased=True, shade=True)
    fig.canvas.draw()
    bodies_buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    bodies_img = Image.fromarray(bodies_buf)

    # composite bodies on top -- they are the focal point of the metaphor and
    # must always read clearly, regardless of matplotlib's approximate
    # (and occasionally unreliable) cross-collection depth sorting.
    lattice_img.alpha_composite(bodies_img)
    return lattice_img


# ----------------------------------------------------------------------------
# 8. THE FOURTH DIMENSION -- a small spacetime (space-vs-time) diagram
#    Two spatial worldlines climbing a vertical TIME axis: the classic,
#    elegant way of showing that time is simply another axis of the same
#    four-dimensional fabric the grid above is bending.
# ----------------------------------------------------------------------------

ST_PANEL_DPI = 150


def render_spacetime_diagram(t_now, t_window_start, t_window_end, w_px, h_px):
    fig = plt.figure(figsize=(w_px / ST_PANEL_DPI, h_px / ST_PANEL_DPI), dpi=ST_PANEL_DPI)
    fig.patch.set_alpha(0)
    ax = fig.add_axes([0.20, 0.20, 0.76, 0.66])
    ax.set_facecolor((1, 1, 1, 0))

    taus = np.linspace(t_window_start, max(t_window_start + 0.001, t_now), 160)
    xa = [body_a_position(tt)[0] for tt in taus]
    xb_raw = [body_b_position(tt) for tt in taus]
    xb = [v[0] if v is not None else None for v in xb_raw]

    ax.plot(xa, taus, color=BODY_A_COLOR, linewidth=2.6, solid_capstyle="round")
    if any(v is not None for v in xb):
        xb_clean = [v for v in xb if v is not None]
        taus_b = [tt for tt, v in zip(taus, xb) if v is not None]
        ax.plot(xb_clean, taus_b, color=BODY_B_COLOR, linewidth=2.6, solid_capstyle="round")

    ax.axhline(t_now, color=INK_DARK, linewidth=1.1, linestyle=(0, (4, 3)), alpha=0.75)
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(t_window_start, t_window_end)

    for spine in ax.spines.values():
        spine.set_color(INK_DARK)
        spine.set_linewidth(1.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])

    fp = fm.FontProperties(family=HAND_FONT)
    ax.set_xlabel("space", fontproperties=fp, fontsize=14, color=INK_DARK, labelpad=2)
    ax.set_ylabel("time", fontproperties=fp, fontsize=14, color=INK_DARK, labelpad=2)
    fig.text(0.5, 0.92, "spacetime (4D)", fontproperties=fp, fontsize=13,
              color=INK_MED, ha="center", va="center")

    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba()).copy()
    plt.close(fig)
    return Image.fromarray(buf)


# ----------------------------------------------------------------------------
# 9. LAYOUT -- fixed, non-overlapping screen regions (1080 x 1920)
# ----------------------------------------------------------------------------

MARGIN = 40
HEADER_XY = (MARGIN, 26)
HEADER_WH = (FRAME_W - 2 * MARGIN, 130)

MAIN_TOP = HEADER_XY[1] + HEADER_WH[1] + 16
MAIN_XY = (0, MAIN_TOP)                         # full width, central to the frame

INFO_TOP = MAIN_TOP + MAIN_REGION_H + 14
INFO_WH = ((FRAME_W - 3 * MARGIN) // 2, 215)
INFO_LEFT_XY = (MARGIN, INFO_TOP)
INFO_RIGHT_XY = (MARGIN * 2 + INFO_WH[0], INFO_TOP)

CAPTION_TOP = INFO_TOP + INFO_WH[1] + 14
CAPTION_WH = (FRAME_W - 2 * MARGIN, 135)

WATERMARK_HEIGHT = int(FRAME_H * 0.024)         # comfortably under the 5% ceiling

TITLE_SIZE = 60
EQUATION_SIZE = 44
NOTE_SIZE = 28
CAPTION_SIZE = 50
HOOK_TITLE_SIZE = 90
HOOK_SUB_SIZE = 42
CREDIT_SIZE = 62
CREDIT_SUB_SIZE = 40

# ----------------------------------------------------------------------------
# 10. SCENE SCRIPT -- the words of the film
# ----------------------------------------------------------------------------

SCENE_TITLE = {
    "flat_space": "Flat Spacetime",
    "single_mass": "Mass Curves Space",
    "moving_mass_waves": "Motion Makes Waves",
    "binary_system": "Two Bodies, One Dance",
}

SCENE_CAPTION = {
    "flat_space": "Empty space. No mass. No curvature.",
    "single_mass": "Drop in a mass, and the grid bends toward it.",
    "moving_mass_waves": "An accelerating mass makes spacetime itself ripple.",
    "binary_system": "Two masses, orbiting -- the ripples grow stronger.",
}

SCENE_EQUATION = {
    "flat_space": r"$\Phi = 0$",
    "single_mass": r"$\Phi(r) = -\dfrac{GM}{\sqrt{r^{2}+a^{2}}}$",
    "moving_mass_waves": r"$h(r,t)\,\sim\,\dfrac{2G}{c^{4}r}\,\ddot{Q}(t)$",
    "binary_system": r"$h_{+}\,\sim\,\cos(2\phi-2\omega t)$",
}

SCENE_EQUATION_NOTE = {
    "flat_space": "no mass, no curvature",
    "single_mass": "curvature grows with mass, fades with distance",
    "moving_mass_waves": "an accelerating mass radiates outgoing ripples",
    "binary_system": "an orbiting pair: a rotating quadrupole wave",
}

SCENE_ANALOGY = {
    "single_mass": "Picture a marble dimpling a stretched net.",
    "moving_mass_waves": "Like ripples spreading outward across a pond.",
}

HOOK_TITLE = "What If Space\nCould Bend?"
HOOK_SUBTITLE = "a short story about gravity"

CREDIT_LINE_1 = "Made by Mugambi Ndwiga"
CREDIT_LINE_2 = "@craftsandengineering"
CREDIT_LINE_3 = "Link to code is in the Description"


# ----------------------------------------------------------------------------
# 11. CARD BUILDERS -- translucent parchment cards, solid crisp ink text
# ----------------------------------------------------------------------------

def make_card_with_lines(w, h, lines, pad_top=18):
    """lines: list of (text, font_name, size, color, mathtext, gap_after)"""
    card = rounded_card(w, h)
    y = pad_top
    for text, font_name, size, color, mathtext, gap_after in lines:
        txt_img = render_text_cached(text, font_name, size, color, mathtext=mathtext)
        if txt_img.width > w - 2 * MARGIN // 2:
            scale = (w - 40) / txt_img.width
            txt_img = txt_img.resize((int(txt_img.width * scale), int(txt_img.height * scale)))
        paste_centered(card, txt_img, w / 2, y + txt_img.height / 2)
        y += txt_img.height + gap_after
    return card


def build_header_card(scene):
    w, h = HEADER_WH
    card = rounded_card(w, h)
    txt_img = render_text_cached(SCENE_TITLE[scene], HAND_FONT, TITLE_SIZE, INK_DARK)
    if txt_img.width > w - 56:
        scale = (w - 56) / txt_img.width
        txt_img = txt_img.resize((int(txt_img.width * scale), int(txt_img.height * scale)))
    paste_centered(card, txt_img, w / 2, h / 2)
    return card


def build_left_info_card(scene):
    w, h = INFO_WH
    lines = [
        (SCENE_EQUATION[scene], HAND_FONT, EQUATION_SIZE, INK_DARK, True, 14),
        (SCENE_EQUATION_NOTE[scene], HAND_FONT, NOTE_SIZE, INK_MED, False, 0),
    ]
    return make_card_with_lines(w, h, lines, pad_top=22)


def _wrap_text(text, max_chars=22):
    words = text.split()
    lines, cur = [], ""
    for word in words:
        trial = (cur + " " + word).strip()
        if len(trial) > max_chars and cur:
            lines.append(cur)
            cur = word
        else:
            cur = trial
    if cur:
        lines.append(cur)
    return lines


def _fit_and_stack(card, imgs, w, h, gap, side_pad=36, top_pad=16):
    """Scale a stack of text images down (uniformly) so they always fit
    inside the card, then paste them centred -- never lets text overflow
    or crash on a long caption."""
    max_w = max(im.width for im in imgs)
    scale_w = min(1.0, (w - side_pad) / max_w)
    total_h = sum(im.height for im in imgs) + gap * (len(imgs) - 1)
    scale_h = min(1.0, (h - top_pad) / max(total_h, 1))
    scale = min(scale_w, scale_h)
    if scale < 0.999:
        imgs = [im.resize((max(1, int(im.width * scale)), max(1, int(im.height * scale))))
                for im in imgs]
    total_h = sum(im.height for im in imgs) + gap * (len(imgs) - 1)
    y = h / 2 - total_h / 2
    for im in imgs:
        paste_centered(card, im, w / 2, y + im.height / 2)
        y += im.height + gap
    return card


def build_right_info_card_text(scene):
    w, h = INFO_WH
    card = rounded_card(w, h)
    text = SCENE_ANALOGY.get(scene, "Spacetime is the stage on which gravity plays.")
    wrapped = _wrap_text(text, max_chars=17)
    imgs = [render_text_cached(line, HAND_FONT, 34, INK_MED) for line in wrapped]
    return _fit_and_stack(card, imgs, w, h, gap=8)


def build_right_info_card_spacetime(t, t0, t1):
    w, h = INFO_WH
    card = rounded_card(w, h)
    diagram = render_spacetime_diagram(t, t0, t1, w - 20, h - 16)
    paste_centered(card, diagram, w / 2, h / 2)
    return card


def build_caption_card(scene):
    w, h = CAPTION_WH
    card = rounded_card(w, h)
    wrapped = _wrap_text(SCENE_CAPTION[scene], max_chars=32)
    imgs = [render_text_cached(line, HAND_FONT, CAPTION_SIZE, INK_DARK) for line in wrapped]
    return _fit_and_stack(card, imgs, w, h, gap=6)


_HEADER_CARD_CACHE = {s: build_header_card(s) for s in SCENE_TITLE}
_LEFT_CARD_CACHE = {s: build_left_info_card(s) for s in SCENE_TITLE}
_RIGHT_TEXT_CARD_CACHE = {s: build_right_info_card_text(s) for s in SCENE_TITLE}
_CAPTION_CARD_CACHE = {s: build_caption_card(s) for s in SCENE_TITLE}


WATERMARK_FONT_SIZE = 28


def build_watermark():
    txt = render_text_cached(WATERMARK_TEXT, HAND_FONT, WATERMARK_FONT_SIZE, INK_DARK)
    if txt.height > WATERMARK_HEIGHT:
        scale = WATERMARK_HEIGHT / txt.height
        txt = txt.resize((max(1, int(txt.width * scale)), max(1, int(txt.height * scale))))
    out = txt.copy()
    r, g, b, a = out.split()
    a = a.point(lambda v: int(v * 0.6))
    out.putalpha(a)
    return out


WATERMARK_IMG = build_watermark()


def paste_watermark(base):
    x = FRAME_W - WATERMARK_IMG.width - MARGIN
    y = FRAME_H - WATERMARK_IMG.height - int(MARGIN * 0.6)
    paste_xy(base, WATERMARK_IMG, x, y)


# ----------------------------------------------------------------------------
# 12. OPENING OVERLAY & CLOSING CARD
# ----------------------------------------------------------------------------

def _scaled_alpha(img, factor):
    """Return a copy of an RGBA image with its alpha channel scaled by
    `factor` (0..1). factor>=0.999 returns the image unchanged (no copy)."""
    if factor >= 0.999:
        return img
    if factor <= 0.001:
        return Image.new("RGBA", img.size, (0, 0, 0, 0))
    img2 = img.copy()
    r, g, b, a = img2.split()
    a = a.point(lambda v: int(v * factor))
    img2.putalpha(a)
    return img2


def graphic_alpha(t):
    """The main 3-D grid breathes into view at the very start of the film."""
    if t >= GRAPHIC_FADE_IN_DURATION:
        return 1.0
    return smoothstep(t / GRAPHIC_FADE_IN_DURATION)


def hook_text_alpha(t):
    """Opacity of the opening title overlay -- fades in, holds, fades out,
    all within the first few seconds, briefly laid over the grid as it
    breathes into view."""
    if t < HOOK_TEXT_FADE_IN_END:
        return smoothstep(t / HOOK_TEXT_FADE_IN_END)
    if t < HOOK_TEXT_HOLD_END:
        return 1.0
    if t < HOOK_TEXT_FADE_OUT_END:
        return 1.0 - smoothstep((t - HOOK_TEXT_HOLD_END) / (HOOK_TEXT_FADE_OUT_END - HOOK_TEXT_HOLD_END))
    return 0.0


def cards_alpha(t):
    """The header/info/caption cards stay hidden during the opening title,
    then fade in gracefully right as the title finishes fading out."""
    if t < CARDS_FADE_START:
        return 0.0
    if t < CARDS_FADE_START + CARDS_FADE_DURATION:
        return smoothstep((t - CARDS_FADE_START) / CARDS_FADE_DURATION)
    return 1.0


def draw_hook_overlay(bg, t):
    alpha = hook_text_alpha(t)
    if alpha <= 0.001:
        return

    lines = HOOK_TITLE.split("\n")
    imgs = [render_text_cached(line, HAND_FONT, HOOK_TITLE_SIZE, INK_DARK) for line in lines]
    total_h = sum(im.height for im in imgs) + 14 * (len(imgs) - 1)
    y = FRAME_H / 2 - total_h / 2 - 70
    for im in imgs:
        im2 = _scaled_alpha(im, alpha)
        paste_centered(bg, im2, FRAME_W / 2, y + im2.height / 2)
        y += im2.height + 14

    sub_img = render_text_cached(HOOK_SUBTITLE, SIGNATURE_FONT, HOOK_SUB_SIZE, INK_MED)
    sub2 = _scaled_alpha(sub_img, alpha)
    paste_centered(bg, sub2, FRAME_W / 2, y + 46)


def render_closing_frame():
    bg = PARCHMENT_BG.copy()
    line1 = render_text_cached(CREDIT_LINE_1, HAND_FONT, CREDIT_SIZE, INK_DARK)
    line2 = render_text_cached(CREDIT_LINE_2, SIGNATURE_FONT, CREDIT_SIZE, INK_DARK)
    line3 = render_text_cached(CREDIT_LINE_3, HAND_FONT, CREDIT_SUB_SIZE, INK_MED)
    total_h = line1.height + line2.height + line3.height + 66
    y = FRAME_H / 2 - total_h / 2
    paste_centered(bg, line1, FRAME_W / 2, y + line1.height / 2); y += line1.height + 24
    paste_centered(bg, line2, FRAME_W / 2, y + line2.height / 2); y += line2.height + 42
    paste_centered(bg, line3, FRAME_W / 2, y + line3.height / 2)
    paste_watermark(bg)
    return bg


_CLOSING_FRAME = render_closing_frame()


# ----------------------------------------------------------------------------
# 13. FULL FRAME COMPOSITOR -- smooth crossfades, nothing ever cuts abruptly
# ----------------------------------------------------------------------------

def camera_azimuth(t):
    return CAMERA_AZIM_START + CAMERA_AZIM_DRIFT * (t / TOTAL_DURATION)


def _crossfade_card(t, t0, scene, lookup):
    """Crossfade a per-scene card from the previous scene's version into the
    current one over the first CARD_TRANSITION seconds of the scene -- the
    card content never cuts abruptly, it dissolves."""
    cur_img = lookup(scene)
    prev = previous_scene(scene)
    if prev is not None and (t - t0) < CARD_TRANSITION:
        frac = smoothstep((t - t0) / CARD_TRANSITION)
        prev_img = lookup(prev)
        out = Image.new("RGBA", cur_img.size, (0, 0, 0, 0))
        out.alpha_composite(_scaled_alpha(prev_img, 1 - frac))
        out.alpha_composite(_scaled_alpha(cur_img, frac))
        return out
    return cur_img


def render_main_frame(t, scene, t0, t1):
    bg = PARCHMENT_BG.copy()

    X, Y, Zc, bodies, _ = field_and_bodies(t)
    graphic = render_grid_scene(X, Y, Zc, bodies, azim=camera_azimuth(t))
    if graphic.size != (MAIN_REGION_W, MAIN_REGION_H):
        graphic = graphic.resize((MAIN_REGION_W, MAIN_REGION_H))
    graphic = _scaled_alpha(graphic, graphic_alpha(t))
    paste_xy(bg, graphic, MAIN_XY[0], MAIN_XY[1])

    c_alpha = cards_alpha(t)
    if c_alpha > 0.001:
        header_card = _crossfade_card(t, t0, scene, lambda s: _HEADER_CARD_CACHE[s])
        left_card = _crossfade_card(t, t0, scene, lambda s: _LEFT_CARD_CACHE[s])
        caption_card = _crossfade_card(t, t0, scene, lambda s: _CAPTION_CARD_CACHE[s])

        if scene == "binary_system":
            dynamic = build_right_info_card_spacetime(t, t0, t1)
            if (t - t0) < CARD_TRANSITION:
                frac = smoothstep((t - t0) / CARD_TRANSITION)
                prev_img = _RIGHT_TEXT_CARD_CACHE["moving_mass_waves"]
                right_card = Image.new("RGBA", dynamic.size, (0, 0, 0, 0))
                right_card.alpha_composite(_scaled_alpha(prev_img, 1 - frac))
                right_card.alpha_composite(_scaled_alpha(dynamic, frac))
            else:
                right_card = dynamic
        else:
            right_card = _crossfade_card(t, t0, scene, lambda s: _RIGHT_TEXT_CARD_CACHE[s])

        if c_alpha < 0.999:
            header_card = _scaled_alpha(header_card, c_alpha)
            left_card = _scaled_alpha(left_card, c_alpha)
            right_card = _scaled_alpha(right_card, c_alpha)
            caption_card = _scaled_alpha(caption_card, c_alpha)

        paste_xy(bg, header_card, *HEADER_XY)
        paste_xy(bg, left_card, *INFO_LEFT_XY)
        paste_xy(bg, right_card, *INFO_RIGHT_XY)
        paste_xy(bg, caption_card, MARGIN, CAPTION_TOP)

    if t < HOOK_TEXT_FADE_OUT_END:
        draw_hook_overlay(bg, t)

    paste_watermark(bg)
    return bg


_LAST_MAIN_FRAME_CACHE = {}


def _last_main_frame():
    """The final frame of the binary scene, frozen -- used as the outgoing
    image for a smooth dissolve into the closing card."""
    if "frame" not in _LAST_MAIN_FRAME_CACHE:
        t_last = T_BINARY_END - (1.0 / FPS)
        scene, t0, t1 = current_scene(t_last)
        _LAST_MAIN_FRAME_CACHE["frame"] = render_main_frame(t_last, scene, t0, t1)
    return _LAST_MAIN_FRAME_CACHE["frame"]


CLOSING_DISSOLVE = 1.2                  # seconds -- a slow, graceful dissolve


def render_frame(t):
    scene, t0, t1 = current_scene(t)
    if scene == "closing":
        if (t - t0) < CLOSING_DISSOLVE:
            frac = smoothstep((t - t0) / CLOSING_DISSOLVE)
            prev_rgb = _last_main_frame().convert("RGB")
            cur_rgb = _CLOSING_FRAME.convert("RGB")
            blended = Image.blend(prev_rgb, cur_rgb, frac)
            return np.asarray(blended, dtype=np.uint8)
        return np.asarray(_CLOSING_FRAME.convert("RGB"), dtype=np.uint8)
    frame = render_main_frame(t, scene, t0, t1)
    return np.asarray(frame.convert("RGB"), dtype=np.uint8)


# ----------------------------------------------------------------------------
# 14. RENDER THE FILM
# ----------------------------------------------------------------------------

def render_video(output_path=OUTPUT_PATH, fps=FPS, crf=27, preset="medium"):
    n_frames = int(round(TOTAL_DURATION * fps))
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    writer = imageio.get_writer(
        output_path, fps=fps, codec="libx264", macro_block_size=1,
        ffmpeg_params=["-crf", str(crf), "-preset", preset, "-pix_fmt", "yuv420p"],
    )
    t_render_start = __import__("time").time()
    for i in range(n_frames):
        t = i / fps
        writer.append_data(render_frame(t))
        if i % max(1, n_frames // 20) == 0:
            elapsed = __import__("time").time() - t_render_start
            print(f"  frame {i + 1:4d}/{n_frames}  (t={t:5.2f}s)  elapsed={elapsed:5.1f}s")
    writer.close()
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Saved {output_path}  ({size_mb:.2f} MB, {n_frames} frames, {TOTAL_DURATION:.1f}s @ {fps}fps)")
    return output_path, size_mb


if __name__ == "__main__":
    path, size_mb = render_video()
    if size_mb > TARGET_MAX_MB:
        print(f"File is {size_mb:.2f} MB (> {TARGET_MAX_MB} MB target) -- re-encoding with a higher CRF...")
        path, size_mb = render_video(crf=32)
    print(f"FINAL: {path}  ({size_mb:.2f} MB)")

    # ------------------------------------------------------------------------
    # 15. DISPLAY + DOWNLOAD -- plays inline everywhere, download link/button
    #     adapts automatically to the environment (Colab vs. plain Jupyter).
    # ------------------------------------------------------------------------
    from IPython.display import Video, FileLink, display

    display(Video(path, embed=True, html_attributes="controls loop", width=380))

    try:
        from google.colab import files as _colab_files
        print("Running in Colab -- starting download...")
        _colab_files.download(path)
    except ImportError:
        print("Download link:")
        display(FileLink(path))
        print(f"(file saved at: {path})")